# 14 · Model E: set models for the repeated trace sets

*Level: advanced. Guide Section 18. Code: `setmodels.SetPooling`, `setmodels.SetStructured`, `setmodels.combine_sets`.*

**Idea.** A packet takes 10 trace sets at the same settings, and today a fixed rule combines them. A set network **learns** the combination. It
must do two things no fixed rule does together:
* ignore a damaged set;
* keep real variation between sets that the image also sees.

In [1]:
from tutorial_utils import *

from ml import setmodels, structured as st

## 1. Permutation invariance and attention pooling

The sets have no order, so the result must not depend on how they are listed. The **DeepSets** form is

$$F(\{\mathbf c_k\}) = \rho\Bigl(\tfrac1K\textstyle\sum_k \psi(\mathbf c_k)\Bigr),$$

* $\psi$ is a network applied to each set separately;
* the average pools them;
* $\rho$ is a network applied to the pooled result.

**Attention** replaces the plain average by learned weights $a_k = e^{\mathbf v^\top\psi_k}/\sum_l e^{\mathbf v^\top\psi_l}$, so a damaged set can get
weight ≈ 0.

In [2]:
rng = np.random.default_rng(0)
W = rng.normal(size=(13, 6)); psi = lambda c: np.tanh(c @ W)
sets = rng.normal(size=(10, 13)); sets[4] += 4                   # set 4 is "damaged"
pool = lambda S: psi(S).mean(0)
print("same result for any order:", np.allclose(pool(sets), pool(sets[rng.permutation(10)])))
others = np.delete(psi(sets), 4, axis=0).mean(0)
v = -(psi(sets)[4] - others)                                     # a scoring vector that dislikes what makes set 4 different
score = psi(sets) @ v; a = np.exp(score) / np.exp(score).sum()
print("attention weights:", a.round(3), " → set 4 gets", a[4].round(4))

same result for any order: True
attention weights: [0.001 0.197 0.077 0.335 0.    0.089 0.001 0.001 0.212 0.087]  → set 4 gets 0.0


## 2. Why a fixed rule can be wrong: intermittent pickup

A pickup with variance $\sigma_p^2$ is on during a fraction $d$ of the time.
* The 5-minute image sees $\sigma_0^2 + d\,\sigma_p^2$.
* Each 0.2-second set sees either $\sigma_0^2$ or $\sigma_0^2+\sigma_p^2$.

The **mean of the sets' variances** estimates the image's value; a **median** picks "on" or "off".

In [3]:
s0, sp = 1.0, 2.0; res = {"median of sd": [], "mean of sd": [], "mean of variances": []}; true = []
for _ in range(3000):
    d = rng.uniform(0.2, 1); on = rng.random(10) < d
    sd_sets = np.sqrt(s0 ** 2 + on * sp ** 2) * (1 + 0.03 * rng.normal(size=10))
    true.append(np.sqrt(s0 ** 2 + d * sp ** 2))
    res["median of sd"].append(np.median(sd_sets)); res["mean of sd"].append(sd_sets.mean()); res["mean of variances"].append(np.sqrt(np.mean(sd_sets ** 2)))
true = np.array(true)
for k, v_ in res.items():
    r = np.log(np.array(v_) / true); print(f"{k:18s}: bias {100 * r.mean():+5.1f} %, RMS {100 * np.sqrt(np.mean(r ** 2)):4.1f} %")

median of sd      : bias  -8.1 %, RMS 26.7 %
mean of sd        : bias  -5.9 %, RMS 12.6 %
mean of variances : bias  -1.2 %, RMS 10.3 %


The **robust** rules drop values more than 5 robust standard deviations from the median. The robust standard deviation is 1.4826 × the median
absolute deviation (MAD), which equals σ for a Gaussian.

## 3. On the toy CCD (world v4: intermittent pickup, 10 % damaged sets)

Model A is used with the sets combined by each fixed rule, and with the learned set network. There are no amplifier labels (`kind="none"`).

In [4]:
V4 = dict(within_packet_pickup=True, damaged_fraction=0.1)
te, cte = synthetic.make_dataset(60, seed=99, **V4); truth_t = np.log(te[["truth.sigma_e", "truth.lambda", "truth.gain"]].to_numpy())
tr, ctr = synthetic.make_dataset(60, seed=1, **V4); Ytr = data.make_targets(tr)[0]
prep = st.StructuredPrep().fit(tr)
Xtr = dict(prep.transform(tr, ctr), sets=ctr["sets: noise: Allan deviation"]); Xte = dict(prep.transform(te, cte), sets=cte["sets: noise: Allan deviation"])
print("sets per example:", Xtr["sets"].shape[1:], "(10 sets × 13 Allan points)")
for comb in setmodels.COMBINERS:
    m, s = setmodels.SetStructured(comb, kind="none").fit(Xtr, Ytr).predict(Xte)
    print(f"sets combined by {comb:26s}: σe error {100 * np.sqrt(np.mean((m[:, 0] - truth_t[:, 0]) ** 2)):5.1f} % (vs truth)")

sets per example: (10, 13) (10 sets × 13 Allan points)


sets combined by mean                      : σe error  31.1 % (vs truth)


sets combined by median                    : σe error  14.3 % (vs truth)


sets combined by robust mean               : σe error  11.3 % (vs truth)


sets combined by mean of variances         : σe error  39.3 % (vs truth)


sets combined by robust mean of variances  : σe error  18.1 % (vs truth)


sets combined by learned                   : σe error   4.4 % (vs truth)


## When to use it

* **Good:** trace sets that genuinely differ (intermittent pickup, glitches), with ~60 or more labelled packets. In our 5-scan tests it was then
  better and steadier than every fixed rule.
* **Bad:** ~30 packets, where a robust fixed rule is better.

---

**That is every model in the project.** Back to the overview: `README.md` in this folder. The full test results are in
`docs/overleaf_ml_models/ml_models.tex`; level 1 runs all these models on real packets (`lvl1/`).